# Run Pre-trained Models for Bowel Retraction Task

#### Make sure to run the following commands in the terminal before running this notebook (use conda or mamba from miniforge):
````bash
 git clone https://gitlab.com/nct_tso_public/needlethreading_il.git
 conda env create -f environment.yml
 conda activate needle-driving
 git clone https://gitlab.com/nct_tso_public/imitation-learning-toolkit.git
 cd imitation-learning-toolkit
 pip install -e .
 cd ..

````

## Import necessary modules

In [ ]:
from config import load_config
from workspace import DiffusionWorkspace, FlowMatchingWorkspace, ACTWorkspace, TaskWorkspace, PhaseACTWorkspace
from constants import PolicyType
import torch


def _load_model(checkpoint_path: str) -> TaskWorkspace:
    config = load_config(checkpoint_path)
    if config.policy_name == PolicyType.DIFFUSION_POLICY.value:
        workspace = DiffusionWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.FLOW_MATCHING.value:
        workspace = FlowMatchingWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.ACT.value:
        workspace = ACTWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.PHASE_ACT.value:
        workspace = PhaseACTWorkspace(config=config, is_inference=True)
    else:
        raise ValueError(f"Unknown policy type: {config.policy_name}")
    workspace.load_checkpoint(path=f"{checkpoint_path}/latest.pt")
    return workspace


## Prepare the dataset and dataloaders

In [ ]:
import numpy as np
import logging
from constants import PHASE_LABEL_KEY
from dataset.dataloader import EpisodicDataset
from pathlib import Path
from config import PolicyConfig


def get_dataloaders(config: PolicyConfig, custom_dataset_folder: str, custom_batch_size: int = 2):
    """Get the dataloaders for training and validation."""
    zarr_path = str(Path(custom_dataset_folder) / "dataset.zarr")
    print("Using zarr path:", zarr_path)
    
    if not Path(config.zarr_path).exists():
        raise FileNotFoundError(f"Zarr file not found at {zarr_path}.")  
    train_dataset = EpisodicDataset(
        zarr_path=zarr_path,
        sampling_mode=config.sampling_mode,
        pred_horizon=config.pred_horizon,
        obs_horizon=config.obs_horizon,
        image_height=config.image_height,
        image_width=config.image_width,
        image_norm_type=config.image_norm_type,
        depth_norm_type=config.depth_norm_type,
        kinematics_norm_type=config.kinematics_norm_type,
        use_color_augmentations=config.use_color_augmentations,
        use_rotation_augmentations=config.use_rotation_augmentations,
        train=True,
        action_backward_shift=config.action_backward_shift,
        camera_names=config.camera_names,
        predict_in_camera_frame=config.predict_in_camera_frame,
        obs_robot_frame=config.obs_robot_frame,
        obs_camera_frame=config.obs_camera_frame,
        deltas_as_actions=config.deltas_as_actions,
        val_ratio=config.ratio_validation_episodes,
        total_ratio_of_episodes=config.total_ratio_of_episodes,
        max_train_episodes=None,
        seed=config.seed,
        skip_initial_steps=config.skip_initial_steps,
        downsample_step=config.downsample_step,
        promote_sparsity=config.promote_sparsity,
        predict_gripper_action=config.predict_gripper_action,
        task_has_phases=config.task_has_phases
    )
    val_dataset = EpisodicDataset(
        zarr_path=zarr_path,
        sampling_mode=config.sampling_mode,
        pred_horizon=config.pred_horizon,
        obs_horizon=config.obs_horizon,
        image_height=config.image_height,
        image_width=config.image_width,
        image_norm_type=config.image_norm_type,
        depth_norm_type=config.depth_norm_type,
        kinematics_norm_type=config.kinematics_norm_type,
        use_color_augmentations=False,
        use_rotation_augmentations=False,
        train=False,
        action_backward_shift=config.action_backward_shift,
        camera_names=config.camera_names,
        predict_in_camera_frame=config.predict_in_camera_frame,
        obs_robot_frame=config.obs_robot_frame,
        obs_camera_frame=config.obs_camera_frame,
        deltas_as_actions=config.deltas_as_actions,
        val_ratio=config.ratio_validation_episodes,
        total_ratio_of_episodes=config.total_ratio_of_episodes,
        seed=config.seed,
        skip_initial_steps=config.skip_initial_steps,
        downsample_step=config.downsample_step,
        promote_sparsity=config.promote_sparsity,
        predict_gripper_action=config.predict_gripper_action,
        task_has_phases=config.task_has_phases
    )
    normalizer = train_dataset.get_normalizer(recompute_depth_stats=config.recompute_depth_stats, device=torch.device(config.device))

    if config.promote_sparsity:
        val_dataset.sparsity_threshold = train_dataset.sparsity_threshold

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=custom_batch_size, shuffle=config.shuffle,
        num_workers=config.num_workers, pin_memory=True, persistent_workers=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=custom_batch_size, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    gripper_positive_class_weights = None
    if config.predict_gripper_action and config.calculate_gripper_positive_class_weights:
        # if the positive class weights are not calculated, set to 1 because that is the "balance" case
        gripper_positive_class_weights = train_dataset.get_gripper_positive_class_imbalance_weight()
    
    if config.task_has_phases:
        selected_eps = np.where(train_dataset.sampler.episode_mask)[0]
        phase_labels = np.concatenate([train_dataset.replay_buffer.get_episode(i)[PHASE_LABEL_KEY].flatten() for i in selected_eps] if len(selected_eps) > 0 else [np.array([])])
        phase_counts = np.bincount(phase_labels, minlength=5)
        logging.info(f"Train phase distribution: {dict(enumerate(phase_counts.tolist()))}")

        selected_eps_val = np.where(val_dataset.sampler.episode_mask)[0]
        if len(selected_eps_val) > 0:
            phase_labels_val = np.concatenate([val_dataset.replay_buffer.get_episode(i)[PHASE_LABEL_KEY].flatten() for i in selected_eps_val])
            phase_counts_val = np.bincount(phase_labels_val, minlength=5)
            logging.info(f"Validation phase distribution: {dict(enumerate(phase_counts_val.tolist()))}")    

    return train_loader, val_loader, normalizer, gripper_positive_class_weights


## Load a pre-trained model

In [ ]:
checkpoint_path ="/path/to/my/checkpoint/dir" # Change this to your checkpoint directory path
workspace = _load_model(checkpoint_path)


In [ ]:
custom_data_path = "/path/to/my/data/dir"  # Set to the path of the directory containing the .zarr archive.
train_loader, val_loader, normalizer, gripper_positive_class_weights = get_dataloaders(workspace.config, custom_dataset_folder=custom_data_path, custom_batch_size=2)
workspace.gripper_positive_class_weights = gripper_positive_class_weights

## Train or Validate the model

In [ ]:
# Example of how to train the model for one epoch
avg_train_metrics = workspace._train_epoch(train_loader)
print(avg_train_metrics.to_dict())

In [ ]:
# Example of how to validate the model for one epoch
avg_val_metrics = workspace._validate(val_loader)
print(avg_val_metrics.to_dict())

In [ ]:
# Example of how to compute loss on the training and validation set for a single batch
from pytorch_utils import dict_apply

batch= next(iter(train_loader))
batch = dict_apply(batch, lambda x: x.to(workspace.config.device, non_blocking=True))
train_metrics = workspace.policy.compute_loss(batch, is_train=True)
batch = next(iter(val_loader))
batch = dict_apply(batch, lambda x: x.to(workspace.config.device, non_blocking=True))
val_metrics = workspace.policy.compute_loss(batch, is_train=False)
